# French election forecasts — FiftyPlusOne-style model

Replicates the method in *How the FiftyPlusOne 2026 election model works* for France (see `PLAN.md`):
polls model (three-pass average, house effects, lineup now-cast) + fundamentals (Stan softmax DLM),
stacked by out-of-sample precision, with an error meta-model fitted on the model's own backtest mistakes
and 40,000 simulations of both rounds.

This notebook is configuration only; the logic lives in the `forecast` package. Rebuild inputs with
`python -m forecast ingest`, backtests with `python -m forecast backtest` / `legi-backtest`.

In [1]:
import warnings
from datetime import date
import polars as pl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pl.Config.set_tbl_rows(20)

from forecast.presidential.pipeline import run_forecast
from forecast.legislative.pipeline import run_legislative_forecast
from forecast.report import charts
from forecast.backtest.report import write_report
from forecast.report.pdf import write_pdf
from IPython.display import display

## Configuration

In [ ]:
AS_OF = date(2026, 11, 27)   # forecast date: polls published by then
N_SIMS = 40_000
# Lineup and alliance probabilities: data/reference/scenarios_pres_2027.yaml, scenarios_legi.yaml
# Model parameters (with provenance): config/model.yaml

## 2027 presidential election

In [3]:
pres = run_forecast(2027, as_of=AS_OF, n_sims=N_SIMS, write=True)
print(pres.report())

KeyboardInterrupt: 

In [ ]:
charts.presidential_probabilities(pres.candidates); plt.show()
charts.presidential_shares(pres.candidates); plt.show()

In [ ]:
display(pres.matchups.head(10))
# far-right runoff scenario: chance of winning overall, if the far right again falls short of its runoff polls, and if not
pres.far_right.with_columns(pl.col(pl.Float64).round(3))

In [ ]:
pres.lineups.head(10)   # the scenario mixture and whether each lineup has been polled

### Given each centre/right ballot
Each column uses only the simulations that drew that ballot (`centre_right` slot in `scenarios_pres_2027.yaml`). Weighting the columns by their probability gives the headline numbers above. The last row takes the slot's candidates together. Every slot is in `conditional.csv` in the output folder; `charts.conditional_matrix(pres, 'rn', ...)` shows Le Pen vs Bardella.

In [ ]:
for measure in ('p_runoff', 'p_win'):
    charts.conditional_matrix(pres, 'centre_right', measure); plt.show()
# median first-round share (%) when on that ballot
pres.conditional_matrix('centre_right', 'share_q50').with_columns(pl.col(pl.Float64).round(1))

### Turnout
First-round turnout comes from past elections and, once pollsters publish them, abstention estimates. Runoff turnout comes from a model of who stays home, fitted on the runoffs since 1965. The second table shows, per runoff, the first-named candidate's chance of winning in the thirds of simulations where turnout comes in furthest below and above what the first round would normally give, with simulation standard errors (`_se`). Rows with `clear_effect` false show no effect of turnout beyond noise.

In [ ]:
charts.turnout_history(pres); plt.show()
display(pres.turnout.with_columns(pl.col(pl.Float64).round(1)))
# clear_effect false: the low/high difference is within simulation noise or negligible
pres.turnout_matchups.select('matchup', 'p', 'turnout_q50', 'p_first_wins', 'p_first_wins_low', 'p_first_wins_low_se',
                             'p_first_wins_high', 'p_first_wins_high_se', 'share_per_3pts', 'share_per_3pts_se',
                             'clear_effect').with_columns(pl.col(pl.Float64).round(3))

## Next legislative election (date assumed in `scenarios_legi.yaml`)

In [ ]:
legi = run_legislative_forecast(as_of=AS_OF, n_sims=N_SIMS, write=True)
print(legi.report())
charts.legislative_seats(legi.totals); plt.show()

In [ ]:
{name: df for name, df in legi.by_scenario.items()}

## Backtest (strict out-of-sample, 2002–2022)

In [ ]:
archive = pl.read_parquet('data/archive/pres_backtest.parquet')
print(write_report(archive))

In [ ]:
from IPython.display import Image, display
for f in ('error_by_horizon', 'weights_over_time', 'calibration'):
    display(Image(f'data/outputs/backtest_pres/{f}.png'))

## PDF report
Headline numbers, charts and tables from the cells above in one PDF: the presidential forecast, the candidate scenarios and the per-ballot matrices, the legislative forecast and the backtest.

In [ ]:
pdf_path = write_pdf(pres, legi, archive)   # data/outputs/forecast_2027_<AS_OF>.pdf
print(pdf_path)